# Reflection Testcase Workflow

This notebook implements a reflection loop for software testing:
- `qa`: writes and revises test cases
- `senior_qa`: reviews and approves or requests revisions

In [2]:
import getpass
import os

def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"请输入您的 {var}")

_set_if_undefined("DEEPSEEK_API_KEY")

In [3]:
import os
from typing import Annotated, Literal

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_deepseek import ChatDeepSeek
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

/Users/elon/code/agents-bootcamp/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
FAKE_USER_STORY = """
As a registered shopper,
I want to apply one promo code during checkout,
so that I can get a discount on my order total.

Acceptance Criteria:
1. Promo code can be applied only once per order.
2. Expired promo codes are rejected with a clear error message.
3. If subtotal is below $50, promo code SAVE10 is invalid.
4. Discount is shown before payment confirmation.
5. Removing promo code recalculates total immediately.
""".strip()

MAX_REVIEW_ROUNDS = 2
DEFAULT_PROVIDER = os.getenv("REFLECTION_PROVIDER", "deepseek")
DEFAULT_MODEL = os.getenv("REFLECTION_MODEL", "deepseek-chat")

class QAState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    round: int

In [9]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a QA engineer. Generate high-quality software test cases from the user story and latest reviewer feedback. "
            "Return test cases in markdown table format with columns: ID, Scenario, Preconditions, Steps, Expected Result, Priority, Type. "
            "Cover positive, negative, boundary, and regression cases. Keep tests specific and executable.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

senior_qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a senior QA reviewer. Review the QA-generated test cases for completeness, clarity, edge cases, and traceability "
            "to acceptance criteria. Respond in this exact structure:\n"
            "DECISION: APPROVED or REVISION_REQUIRED\n"
            "SUMMARY: one short paragraph\n"
            "GAPS: bullet list\n"
            "ACTION_ITEMS: bullet list."
            "If good enough, use DECISION: APPROVED.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

def build_llm(temperature: float, max_tokens: int):
    if DEFAULT_PROVIDER == "openai":
        return ChatOpenAI(
            model=os.getenv("REFLECTION_MODEL", "gpt-4o-mini"),
            temperature=temperature,
            max_tokens=max_tokens,
            timeout=60,
            max_retries=1,
        )

    return ChatDeepSeek(
        model=DEFAULT_MODEL,
        temperature=temperature,
        max_tokens=max_tokens,
        timeout=100,
        max_retries=1,
    )

qa_chain = qa_prompt | build_llm(temperature=0.4, max_tokens=4096)
senior_qa_chain = senior_qa_prompt | build_llm(temperature=0.2, max_tokens=2048)

In [11]:
def qa_node(state: QAState) -> QAState:
    response = qa_chain.invoke({"messages": state["messages"]})
    return {
        "messages": [AIMessage(content=response.content)],
        "round": state["round"],
    }

def senior_qa_node(state: QAState) -> QAState:
    response = senior_qa_chain.invoke({"messages": state["messages"]})
    return {
        "messages": [HumanMessage(content=response.content)],
        "round": state["round"] + 1,
    }

def route_after_review(state: QAState) -> Literal["qa", "__end__"]:
    last_text = state["messages"][-1].content.lower()

    if "decision: approved" in last_text:
        return "__end__"

    if state["round"] >= MAX_REVIEW_ROUNDS:
        return "__end__"

    return "qa"

def build_graph():
    graph_builder = StateGraph(QAState)
    graph_builder.add_node("qa", qa_node)
    graph_builder.add_node("senior_qa", senior_qa_node)

    graph_builder.add_edge(START, "qa")
    graph_builder.add_edge("qa", "senior_qa")
    graph_builder.add_conditional_edges(
        "senior_qa",
        route_after_review,
        {
            "qa": "qa",
            "__end__": END,
        },
    )

    return graph_builder.compile()

In [12]:
def latest_ai_message(messages: list[BaseMessage]) -> str:
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            return msg.content
    return ""

def latest_review_message(messages: list[BaseMessage]) -> str:
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage) and "decision:" in msg.content.lower():
            return msg.content
    return ""

In [16]:
graph = build_graph()

start_message = HumanMessage(
    content=(
        "Create test cases for this user story.\n\n"
        f"{FAKE_USER_STORY}\n\n"
        "After each review, revise the test cases until approved or max rounds reached."
    )
)

result = graph.invoke({"messages": [start_message], "round": 0})

for idx, msg in enumerate(result["messages"], start=1):
    print(f"\n--- Message {idx} | type={msg.type} ---")
    msg.pretty_print()


--- Message 1 | type=human ---
================================ Human Message =================================

Create test cases for this user story.

As a registered shopper,
I want to apply one promo code during checkout,
so that I can get a discount on my order total.

Acceptance Criteria:
1. Promo code can be applied only once per order.
2. Expired promo codes are rejected with a clear error message.
3. If subtotal is below $50, promo code SAVE10 is invalid.
4. Discount is shown before payment confirmation.
5. Removing promo code recalculates total immediately.

After each review, revise the test cases until approved or max rounds reached.

--- Message 2 | type=ai ---
================================== Ai Message ==================================

# Test Cases for Promo Code Application

| ID | Scenario | Preconditions | Steps | Expected Result | Priority | Type |
|----|----------|---------------|-------|----------------|----------|------|
| TC-001 | Apply a valid promo code to